# Stage 3C - DeepSeek Fine-Tuning (LoRA bf16 + Thinking Chain)
## Model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` | 2-class

Fine-tunes DeepSeek-R1-Distill-14B with **LoRA in bf16** using thinking-chain training data
generated by `stage3_deepseek_generate_thinking_chains.ipynb`.

**Training data format (assistant turn):**
```
<think>
{model-generated reasoning chain}
</think>
Final label: {0 or 1}
```

**Key differences from Qwen fine-tuning:**
- `MAX_SEQ_LEN = 1024` (thinking chains are longer)
- `lr = 5e-5` (more conservative for reasoning model)
- `MAX_NEW_TOKENS = 512` at inference (model needs to generate thinking chain)

Requires: `LLM/deepseek_thinking_chains/train_data_with_thinking_2class.csv`

Saved outputs: predictions CSV, summary JSON, metrics CSV, confusion matrix PNG, LoRA adapter.

In [ ]:
!pip install -q transformers datasets peft trl accelerate \
               pandas scikit-learn matplotlib seaborn tqdm

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
import os
import re
import json
import time
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, mean_squared_error, mean_absolute_error,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================
# Paths and configuration
# ============================================
THINKING_TRAIN_PATH = "/content/drive/MyDrive/Yelp_Project/LLM/deepseek_thinking_chains/train_data_with_thinking_2class.csv"
VAL_PATH            = "/content/drive/MyDrive/Yelp_Project/data/processed/val_data.csv"
TEST_PATH           = "/content/drive/MyDrive/Yelp_Project/data/processed/test_data.csv"
OUTPUT_DIR          = "/content/drive/MyDrive/Yelp_Project/LLM/deepseek_r1_distill_qwen14b_2class_finetune_bf16"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "stars"

TASK_TYPE  = "2_class"
RUN_ID     = "deepseek_r1_distill_qwen14b_2class_finetune_bf16"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

# LoRA — conservative for reasoning model
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

# Training — longer sequences due to thinking chains
MAX_SEQ_LEN  = 1024  # larger than Qwen (thinking chains add ~200-400 tokens)
TRAIN_EPOCHS = 2
BATCH_SIZE   = 4
GRAD_ACCUM   = 4     # effective batch = 16
LR           = 5e-5  # conservative: reasoning model is sensitive to lr
WARMUP_RATIO = 0.10  # longer warmup for stability
LR_SCHEDULER = "cosine"

# Inference — must be large enough for thinking chain + answer
INFER_BATCH_SIZE = 8    # smaller batch: longer outputs need more VRAM
MAX_NEW_TOKENS   = 512  # thinking chain + Final label
MAX_INFER_LEN    = 1024

LABELS         = [0, 1]
DISPLAY_LABELS = ["Negative (0)", "Positive (1)"]

print(f"Run ID         : {RUN_ID}")
print(f"Model          : {MODEL_NAME}")
print(f"MAX_SEQ_LEN    : {MAX_SEQ_LEN}  (training)")
print(f"MAX_INFER_LEN  : {MAX_INFER_LEN} / MAX_NEW_TOKENS: {MAX_NEW_TOKENS}  (inference)")
print(f"LR             : {LR}")
print(f"GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================
# Load datasets
# ============================================
# Training: thinking-chain data (already filtered to correct predictions)
df_train = pd.read_csv(THINKING_TRAIN_PATH)
print(f"Train (thinking chain): {len(df_train)} samples")
print(df_train[["original_stars", "label", "thinking_content"]].head(3).to_string())
print(f"Avg thinking length: {df_train['thinking_content'].str.len().mean():.0f} chars")

# Validation and test: load from processed CSV, apply same 2-class filter
def load_2class(path):
    df = pd.read_csv(path)[[TEXT_COL, LABEL_COL]].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df = df[df[LABEL_COL] != 3].copy()
    df["original_stars"] = df[LABEL_COL]
    df["label"] = df[LABEL_COL].apply(lambda x: 0 if x <= 2 else 1)
    return df

df_val  = load_2class(VAL_PATH)
df_test = load_2class(TEST_PATH)
print(f"Val : {len(df_val)}  Test: {len(df_test)}")

In [ ]:
# ============================================
# Prompt helpers
# ============================================
SYSTEM_MSG = (
    "You are a Yelp review sentiment classifier. "
    "Given a review, reason about its sentiment, then output the label. "
    "0 = Negative (original 1-2 stars), 1 = Positive (original 4-5 stars). "
    "Your LAST line must be exactly: Final label: 0  or  Final label: 1"
)

def build_train_messages(text: str, thinking: str, label: int):
    """Training: assistant turn includes <think> chain + Final label."""
    assistant_content = f"<think>\n{thinking}\n</think>\nFinal label: {label}"
    return [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": f"Review: {text}"},
        {"role": "assistant", "content": assistant_content},
    ]

def build_infer_messages(text: str):
    """Inference: no assistant turn."""
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Review: {text}"},
    ]

def parse_output(text: str):
    text = str(text).strip()
    matches = re.findall(r"Final label:\s*([01])", text, flags=re.IGNORECASE)
    if matches:
        return int(matches[-1]), True
    binary = re.findall(r"\b([01])\b", text)
    if binary:
        return int(binary[-1]), False
    return None, False

In [ ]:
# ============================================
# Load tokenizer
# ============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LEN

print("Vocab size       :", tokenizer.vocab_size)
print("EOS token        :", tokenizer.eos_token)
print("model_max_length :", tokenizer.model_max_length)

In [ ]:
# ============================================
# Build HuggingFace Dataset with thinking chains
# ============================================
def df_train_to_hf_dataset(df):
    records = []
    for _, row in df.iterrows():
        messages = build_train_messages(
            row[TEXT_COL], row["thinking_content"], row["label"]
        )
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        records.append({"text": formatted})
    return Dataset.from_list(records)

def df_val_to_hf_dataset(df):
    # Val has no thinking chains: use empty thinking as placeholder
    records = []
    for _, row in df.iterrows():
        messages = build_train_messages(row[TEXT_COL], "", row["label"])
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        records.append({"text": formatted})
    return Dataset.from_list(records)

train_dataset = df_train_to_hf_dataset(df_train)
val_dataset   = df_val_to_hf_dataset(df_val)

print("Train dataset size:", len(train_dataset))
print("Val   dataset size:", len(val_dataset))
print("\nSample (truncated):")
print(train_dataset[0]["text"][:600])

In [ ]:
# ============================================
# Load base model in bf16 (no quantization, 80GB VRAM)
# ============================================
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ============================================
# Fine-tune with SFTTrainer
# ============================================
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")

steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
warmup_steps = max(1, int(WARMUP_RATIO * steps_per_epoch * TRAIN_EPOCHS))
print(f"steps_per_epoch : {steps_per_epoch}")
print(f"warmup_steps    : {warmup_steps}")

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    lr_scheduler_type=LR_SCHEDULER,
    fp16=False,
    bf16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"Training finished in {train_time:.0f}s ({train_time/60:.1f} min)")

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to: {ADAPTER_DIR}")

In [ ]:
# ============================================
# Reload model for inference
# ============================================
del model
del base_model
torch.cuda.empty_cache()

base_model_inf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model_inf.config.use_cache = True

inf_model = PeftModel.from_pretrained(base_model_inf, ADAPTER_DIR)
inf_model.eval()
print("Fine-tuned model loaded for inference.")

In [ ]:
tokenizer.padding_side = "left"  # decoder-only models need left-padding for batch inference

# ============================================
# Batched inference on test set
# ============================================
all_rows = []
eval_start = time.time()

for start in tqdm(range(0, len(df_test), INFER_BATCH_SIZE), desc="Inference"):
    end = min(start + INFER_BATCH_SIZE, len(df_test))
    batch = df_test.iloc[start:end]

    prompts = []
    for _, row in batch.iterrows():
        messages = build_infer_messages(row[TEXT_COL])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INFER_LEN,
    ).to(inf_model.device)

    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,  # 512: thinking chain + answer
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    for i, raw_text in enumerate(decoded):
        pred, strict_ok = parse_output(raw_text)
        all_rows.append({
            "text"            : batch.iloc[i][TEXT_COL],
            "original_stars"  : int(batch.iloc[i]["original_stars"]),
            "true_label"      : int(batch.iloc[i]["label"]),
            "raw_output"      : raw_text,
            "pred"            : pred,
            "strict_format_ok": strict_ok,
        })

eval_time_seconds = time.time() - eval_start

result_df = pd.DataFrame(all_rows)
pred_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_predictions.csv")
result_df.to_csv(pred_path, index=False, encoding="utf-8")
print(f"Saved predictions to: {pred_path}")
print(f"Eval time: {eval_time_seconds:.0f}s")
print(result_df.head())

In [ ]:
# ============================================
# Evaluation
# ============================================
eval_df = result_df.dropna(subset=["pred"]).copy()
eval_df["pred"] = eval_df["pred"].astype(int)

y_true = eval_df["true_label"].to_numpy()
y_pred = eval_df["pred"].to_numpy()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm       = confusion_matrix(y_true, y_pred, labels=LABELS)
mse      = mean_squared_error(y_true, y_pred)
mae      = mean_absolute_error(y_true, y_pred)

valid_prediction_rate = result_df["pred"].notna().mean()
strict_format_rate    = result_df["strict_format_ok"].mean()
eval_speed            = len(result_df) / eval_time_seconds

report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, digits=4)

summary = {
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "LoRA_Finetune_BF16_ThinkingChain",
    "Model"                : MODEL_NAME,
    "LoRA_r"               : LORA_R,
    "LoRA_alpha"           : LORA_ALPHA,
    "Train_Epochs"         : TRAIN_EPOCHS,
    "Effective_Batch_Size" : BATCH_SIZE * GRAD_ACCUM,
    "Learning_Rate"        : LR,
    "Train_Time(s)"        : float(train_time),
    "Batch_Size"           : INFER_BATCH_SIZE,
    "Accuracy"             : float(accuracy),
    "Macro_F1"             : float(macro_f1),
    "Valid_Prediction_Rate": float(valid_prediction_rate),
    "Strict_Format_Rate"   : float(strict_format_rate),
    "Off_By_1_Acc"         : None,
    "Adj_Error_Ratio"      : None,
    "Mid_Confusion_Ratio"  : None,
    "MSE"                  : float(mse),
    "MAE"                  : float(mae),
    "Eval_Time(s)"         : float(eval_time_seconds),
    "Eval_Speed(s/s)"      : float(eval_speed),
    "Confusion_Matrix"     : cm.tolist(),
    "Classification_Report": report,
}

summary_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metrics_row = pd.DataFrame([{
    k: v for k, v in summary.items()
    if k not in ("Confusion_Matrix", "Classification_Report")
}])
metrics_csv_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_metrics.csv")
metrics_row.to_csv(metrics_csv_path, index=False, encoding="utf-8")

print("Saved summary JSON to:", summary_path)
print("Saved metrics CSV to :", metrics_csv_path)
metrics_row

In [ ]:
# ============================================
# Confusion matrix
# ============================================
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=DISPLAY_LABELS, yticklabels=DISPLAY_LABELS)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix - {RUN_ID}")
plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_confusion_matrix.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", fig_path)

In [ ]:
# ============================================
# Final console summary
# ============================================
print(f"Run_ID               : {RUN_ID}")
print(f"Mode                 : LoRA_Finetune_BF16_ThinkingChain")
print(f"Model                : {MODEL_NAME}")
print(f"LR / Epochs          : {LR} / {TRAIN_EPOCHS}")
print(f"Train_Time(s)        : {train_time:.0f}")
print(f"Accuracy             : {accuracy:.4f}")
print(f"Macro_F1             : {macro_f1:.4f}")
print(f"Valid_Prediction_Rate: {valid_prediction_rate:.4f}")
print(f"Strict_Format_Rate   : {strict_format_rate:.4f}")
print(f"MSE                  : {mse:.6f}")
print(f"MAE                  : {mae:.6f}")
print(f"Eval_Time(s)         : {eval_time_seconds:.0f}")